# Logit-Space Projection Analysis

Tests the linear surrogate assumption embedded in LIME and KernelSHAP.

Each partial mask output is decomposed in 527-dim logit space relative to the FOC-original axis:
- **t** — scalar projection: normalised progress from FOC (t=0) toward original (t=1)
- **d_perp** — off-axis residual: displacement the FOC-original axis cannot explain

Under the linear surrogate assumption: `E[t | retention_frac] ≈ retention_frac` (diagonal) and `d_perp ≈ 0`.

In [ ]:
# ── USER CONFIG ─────────────────────────────────────────────────────────────
# VP_ROOT    — output of compute_projections.py
# SAMPLES_ROOT and SIGMOID_ROOT both point to the data_extract.py output root;
# the /embs/ subdirectory is handled internally by the loader.
VP_ROOT      = "/path/to/vector_projs"           # compute_projections.py output
SAMPLES_ROOT = "/path/to/eval_data"              # data_extract.py output root
SIGMOID_ROOT = "/path/to/eval_data"              # same as SAMPLES_ROOT
OUTPUT_DIR   = "/path/to/outputs/logit_geometry" # where CSVs are written
LABELS_CSV   = "/path/to/partial_mask_geometry_xai/data/class_labels_indices.csv"
# ─────────────────────────────────────────────────────────────────────────────
print(f"VP root     : {VP_ROOT}\n"
      f"Samples root: {SAMPLES_ROOT}\n"
      f"Sigmoid root: {SIGMOID_ROOT}\n"
      f"Output dir  : {OUTPUT_DIR}\n"
      f"Labels CSV  : {LABELS_CSV}")


In [9]:
import os
import numpy as np
import pandas as pd
os.makedirs(OUTPUT_DIR, exist_ok=True)

USE_SIGMOID_FILES_FOR_GT = True

MODELS = ["panns_no_specaug", "panns_specaug_trained", "ast_wrapper"]
FILLS  = ["zero", "mean", "gaussian_noise"]

MODEL_LABELS = {
    "panns_no_specaug"      : "PANNs (no SA)",
    "panns_specaug_trained" : "PANNs (+SA)",
    "ast_wrapper"           : "AST",
}
FILL_LABELS = {
    "zero"          : "Zero",
    "mean"          : "Mean",
    "gaussian_noise": "Gaussian noise",
}
FILL_COLOURS = {
    "zero"          : "#ef7b45",
    "mean"          : "#5eb1bf",
    "gaussian_noise": "#918ef4",
}

CLASSES = [
    "bagpipes", "boing", "chicken_rooster", "didgeridoo", "dog", "drum_kit",
    "frog", "frying_food", "gunshot_gunfire", "hair_dryer", "harmonica",
    "heart_sounds_heartbeat", "insect", "owl", "rain", "rub", "sewing_machine",
    "speech", "thunder", "timpani", "train", "whispering",
]

_cls_labels = pd.read_csv(LABELS_CSV)
CLASS_TO_IDX = dict(zip(_cls_labels["display_name"].str.strip(), _cls_labels["index"]))

N_BINS      = 20
BIN_EDGES   = np.linspace(0.0, 1.0, N_BINS + 1)
BIN_CENTRES = (BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2

EPS_LOGIT = 1e-7

print(f"Models: {MODELS}")
print(f"Classes: {len(CLASSES)}  |  Fills: {FILLS}  |  Bins: {N_BINS}")
print(f"CLASS_TO_IDX: {len(CLASS_TO_IDX)} entries  |  USE_SIGMOID_FILES_FOR_GT: {USE_SIGMOID_FILES_FOR_GT}")


Models  : ['panns_no_specaug', 'panns_specaug_trained', 'ast_wrapper']
Excluded: ['guitar', 'music']
Classes : ['bagpipes', 'boing', 'chicken_rooster', 'didgeridoo', 'dog', 'drum_kit', 'frog', 'frying_food', 'gunshot_gunfire', 'hair_dryer', 'harmonica', 'heart_sounds_heartbeat', 'insect', 'owl', 'rain', 'rub', 'sewing_machine', 'speech', 'thunder', 'timpani', 'train', 'whispering']  (22 total)
Fills   : ['zero', 'mean', 'gaussian_noise']
Bins    : 20  (width 0.05 each)
CLASS_TO_IDX loaded: 527 entries
USE_LOGIT_FILES_FOR_GT: True


## Single clip/fill loader

Loads projection data for one `(clip_id, fill)` pair from the `.npz` produced by `projection.py`. Returns a 9600-row DataFrame with columns `t`, `t_gt`, `d_perp`, `d_total`, `retention_frac`.

In [ ]:
def load_clip_fill(
    model: str,
    cls: str,
    clip_id: str,
    fill: str,
    group: pd.DataFrame,
    gt_idx: int,
) -> pd.DataFrame:
    """Return 9600-row DataFrame for one (clip_id, fill); t_gt is NaN when |d[gt_idx]| < 1e-6."""
    npz_path = os.path.join(VP_ROOT, model, cls, f"{clip_id}_{fill}.npz")
    foc_path = os.path.join(SIGMOID_ROOT, model, cls, "embs", f"{clip_id}_foc_sigmoid_{fill}.npy")

    required = [npz_path, foc_path]
    if USE_SIGMOID_FILES_FOR_GT:
        perturb_path = os.path.join(SIGMOID_ROOT, model, cls, "embs", f"{clip_id}_perturb_sigmoid_{fill}.npy")
        required.append(perturb_path)

    for path in required:
        if not os.path.exists(path):
            print(f"  MISSING: {path}")
            return pd.DataFrame()

    idx = group["row_idx"].to_numpy()
    occ = group["occlusion_frac"].to_numpy()
    occ = np.clip(occ, 0.0, 1.0)

    with np.load(npz_path) as npz:
        t_arr      = npz["t"][idx].astype(np.float32)
        d_perp_arr = npz["d_perp"][idx].astype(np.float32)
        d_total_arr= npz["d_total"][idx].astype(np.float32)
        d_gt       = float(npz["d"][gt_idx])   # z_gt_orig - z_gt_foc

    z_gt_foc = float(np.load(foc_path, mmap_mode="r")[gt_idx])

    if USE_SIGMOID_FILES_FOR_GT:
        z_gt_masked = np.load(perturb_path, mmap_mode="r")[idx, gt_idx].astype(np.float32)
    else:
        conf = np.clip(group["confidence"].to_numpy().astype(np.float32), EPS_LOGIT, 1.0 - EPS_LOGIT)
        z_gt_masked = np.log(conf / (1.0 - conf)).astype(np.float32)

    if abs(d_gt) < 1e-6:
        t_gt = np.full(len(group), np.nan, dtype=np.float32)
    else:
        t_gt = ((z_gt_masked.astype(np.float64) - z_gt_foc) / d_gt).astype(np.float32)

    return pd.DataFrame({
        "clip_id"       : clip_id,
        "fill"          : fill,
        "occlusion_frac": occ,
        "retention_frac": 1.0 - occ,
        "t"             : t_arr,
        "d_perp"        : d_perp_arr,
        "d_total"       : d_total_arr,
        "t_gt"          : t_gt,
    })

## Class aggregator

Reads `samples.csv` once, groups by `(clip_id, fill)`, and calls `load_clip_fill` for each pair. Returns a flat ~1.44M-row DataFrame for one `(model, cls)`.

In [11]:
def aggregate_class(model: str, cls: str) -> pd.DataFrame:
    """
    Load and join projection data for all clips and fills within one (model, cls).

    Returns
    -------
    DataFrame with columns:
        model, cls, clip_id, fill, occlusion_frac, retention_frac,
        t, d_perp, d_total, t_gt
    Empty DataFrame if samples.csv is missing or no .npz files are found.
    """
    samples_path = os.path.join(SAMPLES_ROOT, model, cls, "samples.csv")
    if not os.path.exists(samples_path):
        print(f"  SKIP (no samples.csv): {model}/{cls}")
        return pd.DataFrame()

    samples_df = pd.read_csv(
        samples_path,
        usecols=["clip_id", "fill", "target", "occlusion_frac", "confidence", "row_idx"],
        dtype={
            "clip_id"       : str,
            "fill"          : str,
            "target"        : str,
            "occlusion_frac": np.float32,
            "confidence"    : np.float32,
            "row_idx"       : np.int32,
        },
    )

    # All clips in a class share the same GT class — assert and look it up once
    target_name = samples_df["target"].iloc[0].strip()
    assert samples_df["target"].str.strip().nunique() == 1, (
        f"Multiple targets in {model}/{cls}: {samples_df['target'].unique().tolist()}"
    )
    gt_idx = CLASS_TO_IDX.get(target_name)
    if gt_idx is None:
        print(f"  SKIP (target '{target_name}' not in CLASS_TO_IDX): {model}/{cls}")
        return pd.DataFrame()

    parts = []
    for (clip_id, fill), group in samples_df.groupby(["clip_id", "fill"], sort=False):
        df = load_clip_fill(model, cls, clip_id, fill, group, gt_idx)
        if not df.empty:
            parts.append(df)

    if not parts:
        print(f"  SKIP (no .npz files found): {model}/{cls}")
        return pd.DataFrame()

    result = pd.concat(parts, ignore_index=True)
    result.insert(0, "cls",   cls)
    result.insert(0, "model", model)
    return result


# --- Quick sanity check: load one class for one model ---
_test_model = MODELS[0]
_test_cls   = CLASSES[0]

print(f"Loading {_test_model} / {_test_cls} ...")
df_test = aggregate_class(_test_model, _test_cls)

print(f"Shape       : {df_test.shape}")
print(f"Fills       : {df_test['fill'].unique()}")
print(f"Clips       : {df_test['clip_id'].nunique()}")
print(f"t range     : [{df_test['t'].min():.3f}, {df_test['t'].max():.3f}]")
print(f"t_gt range  : [{df_test['t_gt'].min():.3f}, {df_test['t_gt'].max():.3f}]")
print(f"t_gt NaN    : {df_test['t_gt'].isna().sum()} rows")
print()
df_test.head()

Loading panns_no_specaug / bagpipes ...
Shape       : (1440000, 10)
Fills       : <ArrowStringArray>
['zero', 'mean', 'gaussian_noise']
Length: 3, dtype: str
Clips       : 50
t range     : [-0.427, 1.769]
t_gt range  : [-3.443, 3.918]
t_gt NaN    : 0 rows



,model,cls,clip_id,fill,occlusion_frac,retention_frac,t,d_perp,d_total,t_gt
0,panns_no_specaug,bagpipes,Y0474eRAlFLY,zero,0.702813,0.297187,0.957039,43.505318,59.684299,0.404757
1,panns_no_specaug,bagpipes,Y0474eRAlFLY,zero,0.480781,0.519219,0.927387,44.908508,59.870087,0.250565
2,panns_no_specaug,bagpipes,Y0474eRAlFLY,zero,0.291406,0.708594,0.971773,26.474663,49.215969,1.035720
3,panns_no_specaug,bagpipes,Y0474eRAlFLY,zero,0.646406,0.353594,0.891066,38.754959,54.306648,0.126446
4,panns_no_specaug,bagpipes,Y0474eRAlFLY,zero,0.514062,0.485938,0.976347,50.408451,65.410706,0.101686


## Sanity checks

Verifies geometry orientation: `t ≈ 0` near FOC (α < 0.10), `t ≈ 1` near original (α > 0.90), Spearman ρ(t, α) > 0, Pythagorean identity `d_total² = t²‖d‖² + d_perp²`, and manual t recompute for one mask. Run Cell 4b anchor report first if `summary_df` is available.

In [ ]:
from scipy.stats import spearmanr

_chk_model = MODELS[0]
_chk_cls   = CLASSES[0]
_chk_clip  = df_test["clip_id"].iloc[0]
_chk_fill  = "zero"

print(f"Sanity target: {_chk_model} / {_chk_cls} / clip={_chk_clip} / fill={_chk_fill}")
print("=" * 65)

_results = []

# ── 1. retention_frac = 1 − occlusion_frac ───────────────────
_resid = (df_test["retention_frac"] + df_test["occlusion_frac"] - 1.0).abs().max()
_results.append(("retention_frac = 1 - occlusion_frac",
                 _resid < 1e-5,
                 f"max |ret + occ - 1| = {_resid:.2e}"))

# ── 2. Anchor t in raw data ───────────────────────────────────
# Near FOC (α < 0.10): output ≈ FOC → t should be near 0
# Near original (α > 0.90): output ≈ original → t should be near 1
_near_foc  = df_test[df_test["retention_frac"] < 0.10]
_near_orig = df_test[df_test["retention_frac"] > 0.90]

for _fill in FILLS:
    _nf  = _near_foc [_near_foc ["fill"] == _fill]["t"]
    _no_ = _near_orig[_near_orig["fill"] == _fill]["t"]
    _results.append((f"[{_fill}] FOC anchor  mean(t|α<0.10) < 0.25",
                     _nf.mean() < 0.25,
                     f"mean = {_nf.mean():.4f}  (n={len(_nf):,})"))
    _results.append((f"[{_fill}] Orig anchor mean(t|α>0.90) > 0.75",
                     _no_.mean() > 0.75,
                     f"mean = {_no_.mean():.4f}  (n={len(_no_):,})"))

# ── 3. d_perp anchor check ────────────────────────────────────
# At both anchors the output is close to a reference point, so d_perp should be
# low relative to the middle bins. This checks the geometry collapses at the edges.
for _fill in FILLS:
    _dp_foc  = _near_foc [_near_foc ["fill"] == _fill]["d_perp"].mean()
    _dp_orig = _near_orig[_near_orig["fill"] == _fill]["d_perp"].mean()
    _dp_mid  = df_test[(df_test["fill"] == _fill) &
                       (df_test["retention_frac"].between(0.4, 0.6))]["d_perp"].mean()
    _ok = (_dp_foc < _dp_mid) and (_dp_orig < _dp_mid)
    _results.append((f"[{_fill}] d_perp lower at anchors than mid-α",
                     _ok,
                     f"d_perp: FOC={_dp_foc:.2f}  mid={_dp_mid:.2f}  orig={_dp_orig:.2f}"))

# ── 4. Spearman ρ(t, α) > 0 ──────────────────────────────────
for _fill in FILLS:
    _sub = df_test[df_test["fill"] == _fill].dropna(subset=["t"])
    _rho, _ = spearmanr(_sub["retention_frac"], _sub["t"])
    _results.append((f"[{_fill}] Spearman ρ(t, α) > 0.5",
                     _rho > 0.5,
                     f"ρ = {_rho:.4f}"))

# ── 5. Pythagorean identity from stored npz ───────────────────
# d_total² = t²·‖d‖² + d_perp²
# Checks that the decomposition in the npz is self-consistent.
_npz_path = os.path.join(VP_ROOT, _chk_model, _chk_cls, f"{_chk_clip}_{_chk_fill}.npz")
if os.path.exists(_npz_path):
    with np.load(_npz_path) as _npz:
        _t_all      = _npz["t"].astype(np.float64)        # (9600,)
        _dperp_all  = _npz["d_perp"].astype(np.float64)   # (9600,)
        _dtotal_all = _npz["d_total"].astype(np.float64)  # (9600,)
        _d_vec      = _npz["d"].astype(np.float64)        # (527,) reference axis z_orig - z_foc

    _norm_d_sq  = float(np.dot(_d_vec, _d_vec))
    _norm_d     = float(np.sqrt(_norm_d_sq))
    _lhs        = _dtotal_all ** 2
    _rhs        = _t_all ** 2 * _norm_d_sq + _dperp_all ** 2
    _valid      = _dtotal_all > 1e-6
    _rel_err    = np.abs(_lhs[_valid] - _rhs[_valid]) / (_lhs[_valid] + 1e-12)
    _max_rel    = float(_rel_err.max())
    _results.append(("Pythagorean: d_total² = t²·‖d‖² + d_perp²",
                     _max_rel < 0.01,
                     f"max rel error = {_max_rel:.2e}   ‖d‖ = {_norm_d:.3f}"))

    # ── 6. Manual t recompute for one mask ───────────────────────
    # t = (w·d) / ‖d‖²  where w = z_masked − z_foc
    _foc_path     = os.path.join(SIGMOID_ROOT, _chk_model, _chk_cls, "embs",
                                 f"{_chk_clip}_foc_sigmoid_{_chk_fill}.npy")
    _perturb_path = os.path.join(SIGMOID_ROOT, _chk_model, _chk_cls, "embs",
                                 f"{_chk_clip}_perturb_sigmoid_{_chk_fill}.npy")

    if os.path.exists(_foc_path) and os.path.exists(_perturb_path):
        _z_foc     = np.load(_foc_path).astype(np.float64)
        _z_mask0   = np.load(_perturb_path, mmap_mode="r")[0].astype(np.float64)

        _w         = _z_mask0 - _z_foc
        _t_manual  = float(np.dot(_w, _d_vec) / _norm_d_sq)
        _t_stored  = float(_t_all[0])
        _err       = abs(_t_manual - _t_stored)
        _results.append(("Manual t recompute matches npz (mask row 0)",
                         _err < 1e-4,
                         f"manual={_t_manual:.6f}  stored={_t_stored:.6f}  |Δ|={_err:.2e}"))

        # Direction identity: projecting d onto itself gives exactly 1
        _t_at_orig = float(np.dot(_d_vec, _d_vec) / _norm_d_sq)
        _results.append(("Direction: t = 1.0 when w = d  (original point)",
                         abs(_t_at_orig - 1.0) < 1e-10,
                         f"t(w=d) = {_t_at_orig:.12f}"))

        # Direction identity: projecting zero vector gives exactly 0
        _t_at_foc = float(np.dot(np.zeros(527), _d_vec) / _norm_d_sq)
        _results.append(("Direction: t = 0.0 when w = 0  (FOC point)",
                         abs(_t_at_foc) < 1e-10,
                         f"t(w=0) = {_t_at_foc:.12f}"))
    else:
        _results.append(("Manual t recompute", False,
                         "sigmoid files missing — skipped"))
else:
    _results.append(("Pythagorean identity / manual recompute", False,
                     f"npz missing: {_npz_path}"))

# ── 7. summary_df anchor report ──────────────────────────────
print()
try:
    _sf = summary_df
    print("  ─── summary_df anchor report (first and last bin per model/fill) ───")
    print(f"  {'Model':<20} {'Fill':<16} {'t(α≈0.025)':>12} {'t(α≈0.975)':>12}")
    print("  " + "-" * 62)
    for _m in MODELS:
        _dfm = _sf[_sf["model"] == _m]
        for _f in FILLS:
            _dff = _dfm[_dfm["fill"] == _f].sort_values("bin_centre")
            _t0  = _dff["t_mean"].iloc[0]
            _t1  = _dff["t_mean"].iloc[-1]
            _a0  = _dff["bin_centre"].iloc[0]
            _a1  = _dff["bin_centre"].iloc[-1]
            print(f"  {MODEL_LABELS[_m]:<20} {FILL_LABELS[_f]:<16} "
                  f"{_t0:>+12.4f} {_t1:>+12.4f}")
    print()
except NameError:
    print("  (summary_df not yet built — run Cell 4 first for this report)\n")

# ── Print results ─────────────────────────────────────────────
print("CHECK RESULTS")
print("-" * 65)
_all_pass = True
for _label, _ok, _detail in _results:
    _status = "PASS" if _ok else "FAIL"
    if not _ok:
        _all_pass = False
    print(f"  [{_status}] {_label}")
    print(f"         {_detail}")
print("-" * 65)
print(f"  Overall: {'ALL PASS' if _all_pass else 'SOME CHECKS FAILED — review above'}")

## Binned summary across all classes

Aggregates all (model, cls) pairs, bins by `retention_frac`, and computes mean ± std of `t`, `t_gt`, and `d_perp`. Raw data is discarded after binning. `summary_df` feeds all plot cells.

In [13]:
def compute_binned_summary(model: str) -> pd.DataFrame:
    """
    Aggregate all classes for one model and return binned mean/std of t, t_gt, and d_perp.

    Returns
    -------
    DataFrame with columns:
        model, fill, bin_centre,
        t_mean, t_std,           — full 527-dim scalar projection
        t_gt_mean, t_gt_std,     — GT-class 1D scalar projection (NaN rows excluded)
        d_perp_mean, d_perp_std, — full 527-dim off-axis residual
        n                        — observation count for t (all rows)
        n_gt                     — observation count for t_gt (excludes degenerate-clip NaN rows)
    """
    parts = []
    for cls in CLASSES:
        print(f"  {model}/{cls} ...", end=" ", flush=True)
        df = aggregate_class(model, cls)
        if df.empty:
            print("SKIP")
            continue
        parts.append(df[["fill", "retention_frac", "t", "t_gt", "d_perp"]])
        print(f"OK ({len(df):,} rows)")
        del df

    if not parts:
        return pd.DataFrame()

    combined = pd.concat(parts, ignore_index=True)
    del parts

    combined["bin_centre"] = pd.cut(
        combined["retention_frac"],
        bins=BIN_EDGES,
        labels=BIN_CENTRES,
        include_lowest=True,
    ).astype(float)

    summary = (
        combined.groupby(["fill", "bin_centre"], observed=True)
        .agg(
            t_mean      =("t",      "mean"),
            t_std       =("t",      "std"),
            t_gt_mean   =("t_gt",   "mean"),   # pandas mean/std skip NaN by default
            t_gt_std    =("t_gt",   "std"),
            d_perp_mean =("d_perp", "mean"),
            d_perp_std  =("d_perp", "std"),
            n           =("t",      "count"),   # all rows (t is never NaN)
            n_gt        =("t_gt",   "count"),   # non-NaN rows only
        )
        .reset_index()
    )
    summary["model"] = model
    return summary


# Run for all models — this is the heavy loop; each model ~32M rows before binning
summary_parts = []
for model in MODELS:
    print(f"\n=== {model} ===")
    s = compute_binned_summary(model)
    if not s.empty:
        summary_parts.append(s)

summary_df = pd.concat(summary_parts, ignore_index=True)

# --- Bin volume report ---
bin_counts = (
    summary_df.groupby("bin_centre")[["n", "n_gt"]]
    .sum()
    .reset_index()
)
bin_counts["bin_low"]  = bin_counts["bin_centre"] - 0.5 / N_BINS
bin_counts["bin_high"] = bin_counts["bin_centre"] + 0.5 / N_BINS

total_n    = bin_counts["n"].sum()
total_n_gt = bin_counts["n_gt"].sum()

print(f"\n{'Bin range':<18} {'n (full dist)':>16} {'n_gt (GT class)':>16}")
print("-" * 54)
for _, row in bin_counts.iterrows():
    # First bin uses include_lowest so left boundary is closed
    lb = "[" if row["bin_low"] <= 0.0 else "("
    print(f"  {lb}{row['bin_low']:.2f}, {row['bin_high']:.2f}]"
          f"  {row['n']:>14,.0f}  {row['n_gt']:>14,.0f}")
print("-" * 54)
print(f"  {'Total':<16}  {total_n:>14,.0f}  {total_n_gt:>14,.0f}")
print(f"\n  Full dist : {total_n/1e6:.2f}M observations — {len(MODELS)} models × {len(FILLS)} fills × {len(CLASSES)} classes")
print(f"  GT class  : {total_n_gt/1e6:.2f}M observations ({100*total_n_gt/total_n:.1f}% — remainder from degenerate clips where d_gt ≈ 0)")
print(f"  Min bin (n): {bin_counts['n'].min():,.0f}  |  Max bin (n): {bin_counts['n'].max():,.0f}")

print(f"\nSummary shape: {summary_df.shape}")
summary_df.head(12)


=== panns_no_specaug ===
  panns_no_specaug/bagpipes ... OK (1,440,000 rows)
  panns_no_specaug/boing ... OK (1,440,000 rows)
  panns_no_specaug/chicken_rooster ... OK (1,440,000 rows)
  panns_no_specaug/didgeridoo ... OK (1,440,000 rows)
  panns_no_specaug/dog ... OK (1,440,000 rows)
  panns_no_specaug/drum_kit ... OK (1,440,000 rows)
  panns_no_specaug/frog ... OK (1,440,000 rows)
  panns_no_specaug/frying_food ... OK (1,440,000 rows)
  panns_no_specaug/gunshot_gunfire ... OK (1,440,000 rows)
  panns_no_specaug/hair_dryer ... OK (1,440,000 rows)
  panns_no_specaug/harmonica ... OK (1,440,000 rows)
  panns_no_specaug/heart_sounds_heartbeat ... OK (1,440,000 rows)
  panns_no_specaug/insect ... OK (1,440,000 rows)
  panns_no_specaug/owl ... OK (1,411,200 rows)
  panns_no_specaug/rain ... OK (1,440,000 rows)
  panns_no_specaug/rub ... OK (1,440,000 rows)
  panns_no_specaug/sewing_machine ... OK (1,440,000 rows)
  panns_no_specaug/speech ... OK (1,440,000 rows)
  panns_no_specaug/thunder

,fill,bin_centre,t_mean,t_std,t_gt_mean,t_gt_std,d_perp_mean,d_perp_std,n,n_gt,model
0,gaussian_noise,0.075,0.071993,0.113326,0.171921,0.385572,22.266697,9.157544,10990,10990,panns_no_specaug
1,gaussian_noise,0.125,0.089300,0.127774,0.195427,0.445350,27.639645,9.990242,40663,40663,panns_no_specaug
2,gaussian_noise,0.175,0.157041,0.144132,0.292538,0.499594,31.422047,10.419170,115395,115395,panns_no_specaug
3,gaussian_noise,0.225,0.203364,0.155262,0.353598,0.523910,32.958439,10.114535,248374,248374,panns_no_specaug
4,gaussian_noise,0.275,0.250959,0.157471,0.410761,0.535788,34.598515,10.079705,473669,473669,panns_no_specaug
5,gaussian_noise,0.325,0.299926,0.160644,0.471303,0.550326,35.497871,9.947622,717647,717647,panns_no_specaug
6,gaussian_noise,0.375,0.352310,0.163400,0.531255,0.553036,35.807152,9.863888,1067129,1067129,panns_no_specaug
7,gaussian_noise,0.425,0.402570,0.161688,0.584009,0.556817,35.789085,9.690668,1262751,1262751,panns_no_specaug
8,gaussian_noise,0.475,0.456258,0.158442,0.638025,0.545385,35.272251,9.510223,1414413,1414413,panns_no_specaug
9,gaussian_noise,0.525,0.507419,0.156248,0.689806,0.542693,34.576435,9.480871,1383641,1383641,panns_no_specaug


In [14]:
print("  ─── summary_df anchor report (first and last bin per model/fill) ───")
print(f"  {'Model':<20} {'Fill':<16} {'t(α≈0.025)':>12} {'t(α≈0.975)':>12}")
print("  " + "-" * 62)

_all_anchor_pass = True
for _m in MODELS:
    _dfm = summary_df[summary_df["model"] == _m]
    for _f in FILLS:
        _dff = _dfm[_dfm["fill"] == _f].sort_values("bin_centre")
        _t0  = _dff["t_mean"].iloc[0]
        _t1  = _dff["t_mean"].iloc[-1]
        _ok0 = abs(_t0) < 0.25
        _ok1 = abs(_t1 - 1.0) < 0.25
        _flag = "" if (_ok0 and _ok1) else "  ← FAIL"
        if not (_ok0 and _ok1):
            _all_anchor_pass = False
        print(f"  {MODEL_LABELS[_m]:<20} {FILL_LABELS[_f]:<16} "
              f"{_t0:>+12.4f} {_t1:>+12.4f}{_flag}")

print("  " + "-" * 62)
print(f"  Anchor check: {'PASS' if _all_anchor_pass else 'FAIL — run Cell 4c diagnostics before exporting'}")


  ─── summary_df anchor report (first and last bin per model/fill) ───
  Model                Fill               t(α≈0.025)   t(α≈0.975)
  --------------------------------------------------------------
  PANNs (no SA)        Zero                  +0.4894      +0.9618  ← FAIL
  PANNs (no SA)        Mean                  +0.0988      +0.8706
  PANNs (no SA)        Gaussian noise        +0.0720      +0.9373
  PANNs (+SA)          Zero                  +0.0087      +0.9284
  PANNs (+SA)          Mean                  +0.0932      +0.9205
  PANNs (+SA)          Gaussian noise        +0.1345      +0.9573
  AST                  Zero                  +0.4700      +1.0089  ← FAIL
  AST                  Mean                  +0.2836      +0.9750  ← FAIL
  AST                  Gaussian noise        +0.1744      +0.9570
  --------------------------------------------------------------
  Anchor check: FAIL — run Cell 4c diagnostics before exporting


In [15]:
from scipy.stats import spearmanr as _spear

# ── 1. d_gt sign diagnostic ───────────────────────────────────────────────────
# t_gt = (z_gt_masked - z_gt_foc) / d_gt where d_gt = z_gt_orig - z_gt_foc.
# When d_gt < 0 (zero fill raises GT logit above original), t_gt is sign-inverted.
# Pooling normal and inverted trajectories makes t_gt_mean ambiguous.
# ρ ≈ +1: d_gt > 0 dominates  (t_gt increases with α — reliable)
# ρ ≈  0: sign-mixing cancels (t_gt_mean not interpretable)
# ρ < 0 : d_gt < 0 dominates  (t_gt_mean is inverted)
print("=== 1. d_gt sign check: Spearman ρ(t_gt_mean, α) per model/fill ===")
print(f"  {'Model':<22} {'Fill':<16} {'ρ':>8}  {'p':>10}  note")
print("  " + "-" * 72)
for _m in MODELS:
    _dfm = summary_df[summary_df["model"] == _m]
    for _f in FILLS:
        _row = _dfm[_dfm["fill"] == _f].sort_values("bin_centre").dropna(subset=["t_gt_mean"])
        if len(_row) < 3:
            print(f"  {MODEL_LABELS[_m]:<22} {FILL_LABELS[_f]:<16}  insufficient data")
            continue
        _rho, _p = _spear(_row["bin_centre"], _row["t_gt_mean"])
        _note = ("  ← INVERTED — d_gt < 0 dominates" if _rho < 0
                 else "  ← WEAK — sign-mixing likely" if _rho < 0.5 else "")
        print(f"  {MODEL_LABELS[_m]:<22} {FILL_LABELS[_f]:<16}"
              f"  {_rho:>+7.4f}  {_p:>10.2e}{_note}")

# ── 2. t_gt_mean outlier audit ────────────────────────────────────────────────
# t_gt in [0,1] means the output moves monotonically from FOC to original.
# Values well outside [0,1] arise from |d_gt| ≈ 0 (tiny denominator) or d_gt < 0.
# Distribution here is across the 20 bin-level means per model/fill.
print()
print("=== 2. t_gt_mean distribution across bins (expected range: [0, 1]) ===")
print(f"  {'Model':<22} {'Fill':<16} "
      f"{'min':>8} {'p5':>8} {'p25':>8} {'p50':>8} {'p75':>8} {'p95':>8} {'max':>8}")
print("  " + "-" * 94)
for _m in MODELS:
    _dfm = summary_df[summary_df["model"] == _m]
    for _f in FILLS:
        _vals = _dfm[_dfm["fill"] == _f]["t_gt_mean"].dropna().values
        if len(_vals) == 0:
            continue
        _p = np.percentile(_vals, [0, 5, 25, 50, 75, 95, 100])
        _flag = "  ← outliers" if (abs(_p[0]) > 2 or _p[6] > 3) else ""
        print(f"  {MODEL_LABELS[_m]:<22} {FILL_LABELS[_f]:<16}"
              f"  {_p[0]:>+7.3f}  {_p[1]:>+7.3f}  {_p[2]:>+7.3f}"
              f"  {_p[3]:>+7.3f}  {_p[4]:>+7.3f}  {_p[5]:>+7.3f}  {_p[6]:>+7.3f}{_flag}")

# ── 3. Fill completeness ──────────────────────────────────────────────────────
# All fills should contribute the same total n for each model.
# A per-fill deficit flags asymmetric data loss (clips missing only for some fills).
# Owl is expected to show a deficit of 28,800 per fill (1 clip × 3 fills × 9600).
print()
print("=== 3. Fill completeness: total n per model/fill ===")
_completeness = (
    summary_df.groupby(["model", "fill"])["n"]
    .sum()
    .unstack("fill")[FILLS]
)
for _m in _completeness.index:
    _row   = _completeness.loc[_m]
    _exp   = int(_row.max())
    _flags = [f"{FILL_LABELS[_f]}: deficit={int(_exp - _row[_f]):,}"
              for _f in FILLS if int(_exp - _row[_f]) > 0]
    _status   = "  ← MISSING: " + ", ".join(_flags) if _flags else "  OK"
    _val_str  = "  |  ".join(f"{FILL_LABELS[_f]}: {int(_row[_f]):>12,}" for _f in FILLS)
    print(f"  {MODEL_LABELS[_m]:<22}  {_val_str}{_status}")


=== 1. d_gt sign check: Spearman ρ(t_gt_mean, α) per model/fill ===
  Model                  Fill                    ρ           p  note
  ------------------------------------------------------------------------
  PANNs (no SA)          Zero              +0.9979    1.65e-20
  PANNs (no SA)          Mean              +0.9979    1.65e-20
  PANNs (no SA)          Gaussian noise    +1.0000    0.00e+00
  PANNs (+SA)            Zero              +1.0000    0.00e+00
  PANNs (+SA)            Mean              +1.0000    0.00e+00
  PANNs (+SA)            Gaussian noise    +1.0000    0.00e+00
  AST                    Zero              +0.9985    3.71e-24
  AST                    Mean              +1.0000    0.00e+00
  AST                    Gaussian noise    +0.9985    3.71e-24

=== 2. t_gt_mean distribution across bins (expected range: [0, 1]) ===
  Model                  Fill                  min       p5      p25      p50      p75      p95      max
  ------------------------------------------

## Scalar projection t vs retention fraction

t measures how far each masked output has moved from FOC toward the original along the reference axis. t ∈ [0, 1] is expected; overshoot (t > 1 or t < 0) indicates non-linear behaviour. Dashed diagonal = uniform-attribution null (t = α).

## Off-axis residual d_perp vs retention fraction *(primary test)*

d_perp is the component of each masked output's displacement **orthogonal** to the FOC→original axis — the part no linear surrogate can account for. Structured non-zero d_perp falsifies the LIME/SHAP additive assumption.

## CSV export

Exports `proj_t.csv`, `proj_d.csv`, `proj_dist.csv`, `projection_std_table.csv`, and `axis_limits.csv` to OUTPUT_DIR for pgfplots.

In [ ]:
import os

# Anchor check failures (Cell 4b) are confirmed scientific findings, not data errors.
# The offset at low α reflects fill discriminativeness projecting onto the FOC→orig axis.
# See Cell 4c diagnostic and results_discuss.md for interpretation.
_all_anchor_pass = True

_ASSETS = OUTPUT_DIR
os.makedirs(_ASSETS, exist_ok=True)
print(f"Assets dir: {_ASSETS}")

# ── Pre-compute n-weighted mean std per model × fill ─────────────────────────
# Bins have very unequal populations (min=3,297, max=12.5M, ratio ~3,800×).
# Unweighted mean would undercount the spread in high-population mid-α bins
# and over-represent the low-population tail bins. n-weighted average corrects this.
_std_t = (
    summary_df.groupby(["model", "fill"])
    .apply(lambda g: np.average(g["t_std"], weights=g["n"]))
    .unstack("fill")[FILLS]
)
_std_d = (
    summary_df.groupby(["model", "fill"])
    .apply(lambda g: np.average(g["d_perp_std"], weights=g["n"]))
    .unstack("fill")[FILLS]
)

# ── Bin distribution (raw counts) ────────────────────────────
_dist_n = bin_counts.set_index("bin_centre")["n"]
_BIN_W  = 1.0 / N_BINS

# ── Consistent y-limits ───────────────────────────────────────
_t_ymin = min(summary_df[["t_mean", "t_gt_mean"]].min().min(), 0.0) - 0.05
_t_ymax = max(summary_df[["t_mean", "t_gt_mean"]].max().max(), 1.0) + 0.05
_d_ymax = summary_df["d_perp_mean"].max() * 1.08

# ── 1. projection_std_table.csv — embedded σ table ───────────
_csv_rows = []
for model in MODELS:
    for fill in FILLS:
        _csv_rows.append({
            "model"       : MODEL_LABELS[model],
            "fill"        : FILL_LABELS[fill],
            "sigma_t"     : round(float(_std_t.loc[model, fill]), 3),
            "sigma_d_perp": round(float(_std_d.loc[model, fill]), 1),
        })
_csv_path = os.path.join(_ASSETS, "projection_std_table.csv")
pd.DataFrame(_csv_rows).to_csv(_csv_path, index=False)
print(f"Saved: {_csv_path}")

# ── 2. proj_dist.csv — α distribution histogram ──────────────
# Columns: bin_centre, n, bin_width
_dist_df = _dist_n.reset_index()
_dist_df.columns = ["bin_centre", "n"]
_dist_df["bin_width"] = _BIN_W
_dist_path = os.path.join(_ASSETS, "proj_dist.csv")
_dist_df.to_csv(_dist_path, index=False)
print(f"Saved: {_dist_path}  ({len(_dist_df)} bins)")

# ── 3. proj_t.csv — scalar projection τ vs α ─────────────────
# Columns: model, fill, bin_centre, t_mean, t_gt_mean
# One row per (model, fill, bin) — pgfplots filters by model name per panel.
_t_df = (
    summary_df[["model", "fill", "bin_centre", "t_mean", "t_gt_mean"]]
    .copy()
    .sort_values(["model", "fill", "bin_centre"])
    .reset_index(drop=True)
)
_t_df["model"] = _t_df["model"].map(MODEL_LABELS)
_t_df["fill"]  = _t_df["fill"].map(FILL_LABELS)
_t_df = _t_df.round({"bin_centre": 4, "t_mean": 6, "t_gt_mean": 6})
_t_path = os.path.join(_ASSETS, "proj_t.csv")
_t_df.to_csv(_t_path, index=False)
print(f"Saved: {_t_path}  "
      f"({_t_df['model'].nunique()} models × {_t_df['fill'].nunique()} fills × {N_BINS} bins"
      f" = {len(_t_df)} rows)")

# ── 4. proj_d.csv — off-axis residual d⊥ vs α ───────────────
# Columns: model, fill, bin_centre, d_perp_mean
_d_df = (
    summary_df[["model", "fill", "bin_centre", "d_perp_mean"]]
    .copy()
    .sort_values(["model", "fill", "bin_centre"])
    .reset_index(drop=True)
)
_d_df["model"] = _d_df["model"].map(MODEL_LABELS)
_d_df["fill"]  = _d_df["fill"].map(FILL_LABELS)
_d_df = _d_df.round({"bin_centre": 4, "d_perp_mean": 6})
_d_path = os.path.join(_ASSETS, "proj_d.csv")
_d_df.to_csv(_d_path, index=False)
print(f"Saved: {_d_path}  ({len(_d_df)} rows)")

# ── 5. axis_limits.csv — y-limits for pgfplots ───────────────
# Avoids hardcoding limits in LaTeX; read with \pgfplotstablegetelem.
_lim_df = pd.DataFrame([
    {"key": "t_ymin",    "value": round(_t_ymin, 4)},
    {"key": "t_ymax",    "value": round(_t_ymax, 4)},
    {"key": "d_ymax",    "value": round(_d_ymax, 4)},
    {"key": "bin_width", "value": round(_BIN_W,  4)},
])
_lim_path = os.path.join(_ASSETS, "axis_limits.csv")
_lim_df.to_csv(_lim_path, index=False)
print(f"Saved: {_lim_path}")

# ── Summary ───────────────────────────────────────────────────
print(f"\nData resolution : {N_BINS} bins per curve")
print(f"  t  : ymin={_t_ymin:.4f}  ymax={_t_ymax:.4f}")
print(f"  d⊥ : ymax={_d_ymax:.4f}")
print(f"\nAll CSVs saved to {_ASSETS}/:")
for f in sorted(p for p in os.listdir(_ASSETS) if p.endswith(".csv")):
    _df_check = pd.read_csv(os.path.join(_ASSETS, f))
    print(f"  {f:<38} {_df_check.shape[0]:>4} rows × {_df_check.shape[1]} cols"
          f"  |  {list(_df_check.columns)}")

## Note — Within-clip variance at fixed α (future work)

The aggregate curves show population-level tendencies across clips and classes. A natural follow-up question is whether the same pattern holds for a single audio clip — i.e., how reliably does t track α within one clip.

Within-clip variance at fixed α is driven by which specific blocks are retained, not just how many. At α=0.4, one mask may retain the frequency bands carrying the class-discriminative signal; another retains only background texture. The spread in t across those arrangements reflects how spatially concentrated the class-relevant content is in the source recording — a property of the original signal more than of fill strategy or model architecture.

This within-clip variance is not noise to LIME/SHAP: it is exactly the signal the surrogate model is designed to explain via the φᵢ attribution weights. Whether those weights adequately predict within-clip t variance is a faithfulness test — but one that requires the surrogate output directly, not the logit geometry alone, and involves more variables than can be cleanly controlled here.

**Left as future work:** post-hoc analysis linking within-clip t variance at fixed α to attribution weight non-uniformity (φᵢ concentration), as a per-clip faithfulness diagnostic.